# Policy index setup (Search-managed Standard ingestion)

Use the existing Azure AI Search service; this notebook creates no Search service, knowledge base or hosted agent. The agent queries an ordinary index directly.

Pipeline: policy PDFs -> Blob Storage -> Search indexer -> Content Understanding skill -> policy index -> Agent Framework.

Before starting, run `infra\setup.ps1` from the repository root and sign in with `az login`. Select `.tools\policy-venv\Scripts\python.exe` as the notebook's Python environment (your notebook host may prompt to install its kernel support). To use a separately managed kernel, install `infra/requirements-policy-agent-framework.txt` in that environment. Agent Framework is installed through pip, not npm; its core and OpenAI packages have independent version numbers. You need an existing Foundry model deployment and CU claim analyzer. The Search identity needs Storage Blob Data Reader on the policy storage account and Cognitive Services User on Foundry. Setup reports missing permissions but never assigns roles or changes networking.

The notebook uses the same functions as `infra/setup-policy-index.py`. Only the explicit apply cell can upload PDFs or create ingestion objects.

In [ ]:
import importlib.util, json
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'infra' / 'setup-policy-index.py').exists())
spec = importlib.util.spec_from_file_location('policy_index_setup', root / 'infra' / 'setup-policy-index.py')
setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(setup)
http = setup.Http()

# Fresh clone: .tools/policy-agent-framework.json is git-ignored, so seed existing resources explicitly.
# The analyzer hash is read from the existing analyzer; never enter it manually.
SEED = {
    'modelDeployment': 'gpt-5',  # existing Foundry deployment
    'claimAnalyzerId': 'lab664_policy_checks_claim_summary',  # existing CU analyzer
}

## 1. Check sources and prerequisites

The seed names above match this lab environment; replace them with your existing deployment and analyzer names when needed. The analyzer definition is read from CU, so no hash needs to be entered. Preflight validates PDF hashes, citation mappings, resource access and roles. It changes nothing in Azure.

In [ ]:
report = setup.preflight(http, root, seed=SEED)
print(json.dumps({k: report.get(k) for k in ('ready', 'documents', 'names', 'claimAnalyzer', 'missingRoles', 'problems', 'existing')}, indent=2))

## 2. Inspect the managed pipeline

The CU skill uses the Search preview API for semantic chunks, table Markdown, page ranges and model-generated figure descriptions. `contentExtractionMode: standard` is a knowledge-source setting, not an index property; this pipeline configures the CU skill directly instead.

Only registered policy PDFs are ingested. Answer guides, claim packets and editable duplicates aren't policy knowledge. The generation name changes when sources or extraction settings change.

In [ ]:
sources = setup.load_sources(root)
settings = setup.runtime_settings(root, SEED)
generation = setup.generation_hash(sources, settings)
names = setup.names_for(generation)
print(json.dumps({'sourceFiles': [d['file'] for d in sources['documents']],
                  'names': names, 'skillset': setup.skillset_definition(names, generation, settings)}, indent=2))

## 3. Apply explicitly and activate only after verification

Set `APPLY = True` only after resolving the preflight report. This can incur Blob, Search, CU and model charges. Apply checks prerequisites again, uploads the source PDFs, creates generation-scoped objects, waits for ingestion and verifies every source plus state-filtered retrieval. Only then is the workbench configuration switched to the new index. The existing working configuration remains active on failure.

Changed sources automatically create a new generation. `REFRESH = True` resets and reprocesses an existing generation; it isn't necessary for an ordinary source update. No previous Azure resources are deleted.

In [ ]:
APPLY = False
REFRESH = False
if APPLY:
    print(json.dumps(setup.apply(http, root, refresh=REFRESH, seed=SEED), indent=2))
else:
    print('APPLY is False: preflight only')

## 4. Compare the agent results

After successful activation, open the workbench and run the same claim in **No policy retrieval** and **Azure Search retrieval**. The five checks, claim facts, model and question stay the same; only retrieved SOP evidence changes. Inspect Expected/Actual values and citations, and use Chat for follow-ups.

Use Arizona's eligibility figure, Texas's blank table cells and Washington's scanned pages to inspect extraction quality. Figure descriptions are model-generated, not a guarantee of correct interpretation: ambiguous evidence must still require review. The Document, CU Layout and Digital parse tabs are separate cached representations, not the managed index's Standard output.